In [0]:
data = [
    (1, "manish", 20000, "india"),
    (2, "john", 30000, "usa"),
    (3, "sara", 25000, "uk"),
    (4, "ravi", 40000, "india"),
    (5, "amit", 35000, "india"),
    (6, "david", 45000, "usa"),
    (7, "priya", 50000, "india"),
    (8, "tom", 28000, "uk")
]

columns = ["emp_id", "name", "salary", "loc"]

df = spark.createDataFrame(data, columns)

In [0]:
output_path = "/Volumes/databricks-pyspark/databricks-pyspark-schema/internal-v1/output/small_files/"
df.repartition(8).write.mode('overwrite').format("delta").save(output_path)

In [0]:
display(dbutils.fs.ls(output_path))

In [0]:
read_df = spark.read.load(output_path)
read_df.show()

In [0]:
from pyspark.sql.functions import count,spark_partition_id
read_df.withColumn("partition_id", spark_partition_id()).groupBy("partition_id").agg(count("*")).show()

In [0]:
compact_path = "/Volumes/databricks-pyspark/databricks-pyspark-schema/internal-v1/output/compact_files/"
read_df.coalesce(2).write.mode('overwrite').format('delta').save(compact_path)

In [0]:
display(dbutils.fs.ls(compact_path))

In [0]:
verify_df = spark.read.load(compact_path)
verify_df.withColumn("partition_id",spark_partition_id()).groupBy("partition_id").agg(count("*")).show()